# Customer Lifetime Value - Business Metrics

Calculates customer value metrics for RFM analysis (Recency, Frequency, Monetary). Answers:
- Who are the highest-value customers?
- How often do customers purchase?
- When was their last purchase?
- What's the average order value per customer?

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.gold.agg_customer_lifetime_value AS
SELECT 
  cu.customer_id,
  cu.first_name,
  cu.last_name,
  cu.country,
  cu.gender,
  cu.marital_status,
  
  -- Monetary: Lifetime value metrics
  SUM(fs.sales_amount) AS lifetime_value,
  COUNT(DISTINCT fs.order_number) AS total_orders,
  ROUND(AVG(fs.sales_amount), 2) AS avg_order_value,
  SUM(fs.quantity) AS total_items_purchased,
  
  -- Recency: Last purchase date
  MIN(fs.order_date) AS first_order_date,
  MAX(fs.order_date) AS last_order_date,
  DATEDIFF(CURRENT_DATE(), MAX(fs.order_date)) AS days_since_last_order,
  
  -- Frequency: Purchase patterns
  DATEDIFF(MAX(fs.order_date), MIN(fs.order_date)) AS customer_lifespan_days,
  CASE 
    WHEN DATEDIFF(MAX(fs.order_date), MIN(fs.order_date)) > 0 
    THEN ROUND(COUNT(DISTINCT fs.order_number) / DATEDIFF(MAX(fs.order_date), MIN(fs.order_date)) * 30, 2)
    ELSE 0
  END AS avg_orders_per_month

FROM workspace.gold.fact_sales fs
INNER JOIN workspace.gold.dim_customers cu 
  ON fs.customer_key = cu.customer_key
  AND cu.is_current = TRUE

WHERE fs.order_date IS NOT NULL

GROUP BY cu.customer_id, cu.first_name, cu.last_name, 
         cu.country, cu.gender, cu.marital_status

In [0]:
%sql
SELECT 
  CONCAT(first_name, ' ', last_name) AS customer_name,
  country,
  lifetime_value,
  total_orders,
  avg_order_value,
  days_since_last_order
FROM workspace.gold.agg_customer_lifetime_value
ORDER BY lifetime_value DESC
LIMIT 10

In [0]:
%sql
SELECT 
  COUNT(*) AS total_customers,
  SUM(lifetime_value) AS total_revenue,
  ROUND(AVG(lifetime_value), 2) AS avg_lifetime_value,
  ROUND(AVG(total_orders), 2) AS avg_orders_per_customer,
  ROUND(AVG(days_since_last_order), 0) AS avg_days_since_last_order
FROM workspace.gold.agg_customer_lifetime_value